In [1]:
import sys
import os
import json

import datasets
from transformers import AutoTokenizer

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.agents.math_agent import MathAgent

### Agent setup

In [2]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

agent = MathAgent(
    response_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    eval_model_name="gemini-2.0-flash", 
    temperature=0.5,
    tokenizer=tokenizer,
)

### UMWP

In [3]:
dataset_name = "UMWP"
path = f"../../../dataset/raw_model_responses/test/test_{dataset_name}.json"

data = datasets.load_dataset("json", data_files=path)
data = data["train"]

In [ ]:
final_results = []
input_tokens = 0
output_tokens = 0

for sample in data:
    question = sample["additional_info"]["question"]

    corrected_responses = []

    for i in range(len(sample["response"])):
        response = sample["response"][i]
        result = agent.model.invoke({
            "question": question,
            "initial_response": response,
        })
        corrected_responses.append(result["corrected_response"] if "corrected_response" in result else response)
        input_tokens += result["input_tokens"]
        output_tokens += result["output_tokens"]

    sample["response"] = corrected_responses
    final_results.append(sample)

Gemini produced an empty response. Continuing with empty message
Feedback: 
2025-10-22 18:16:32.819 | INFO     | src.agents.math_agent:check_response:71 - Generated correction: 
Gemini produced an empty response. Continuing with empty message
Feedback: 
2025-10-22 18:16:33.420 | INFO     | src.agents.math_agent:check_response:71 - Generated correction: 
Gemini produced an empty response. Continuing with empty message
Feedback: 
2025-10-22 18:16:33.926 | INFO     | src.agents.math_agent:check_response:71 - Generated correction: 
Gemini produced an empty response. Continuing with empty message
Feedback: 
2025-10-22 18:16:34.509 | INFO     | src.agents.math_agent:check_response:71 - Generated correction: 
Gemini produced an empty response. Continuing with empty message
Feedback: 
2025-10-22 18:16:35.085 | INFO     | src.agents.math_agent:check_response:71 - Generated correction: 
Gemini produced an empty response. Continuing with empty message
Feedback: 
2025-10-22 18:16:35.645 | INFO    

In [7]:
print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {output_tokens}")
print(f"Average input tokens: {input_tokens / (len(final_results)*5)}")
print(f"Average output tokens: {output_tokens / (len(final_results)*5)}")

Input tokens: 148690
Output tokens: 39955
Average input tokens: 1416.095238095238
Average output tokens: 380.5238095238095


In [8]:
final_results

[{'task_info': {'dataset': 'UMWP', 'type': 'QA'},
  'additional_info': {'answer': [45.0],
   'answerable': True,
   'domain': 'Math',
   'model': 'Meta-Llama-3.1-8B-Instruct',
   'question': 'Carol was sending out birthday invitations to her friends. If each package of invitations she bought had 9 invitations in it and she bought 5 packs,.how many friends can she invite?',
   'source': 'ASDiv'},
  'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a meticulous AI mathematician. Your task is to solve the following math problem.\n\nFollow these steps carefully:\n1. **Analyze the problem:** First, understand the given information and what is being asked.\n2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.\n3. **Solve or Explain:**\n   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then cl

In [9]:
with open(f"../../../dataset/math_agent_responses/test_math_agent_{dataset_name}.json", "w") as f:
    json.dump(final_results, f, indent=4)